# OpenMontage Stage 1 Colab Check

This notebook performs the checks needed to validate a Colab GPU environment and runs the strict Stage 1 smoke test.
Use a GPU runtime (Runtime -> Change runtime type -> GPU). Run cells in order.


In [ ]:
# 1) Show GPU via nvidia-smi (shell)
!nvidia-smi

In [ ]:
# 2) Check Python and PyTorch CUDA availability
import sys, subprocess
print('Python:', sys.version.replace('
',' '))
try:
    import torch
    print('torch:', torch.__version__)
    print('cuda_available:', torch.cuda.is_available())
    print('cuda_version:', torch.version.cuda)
except Exception as e:
    print('torch import failed:', e)
# Show sys.path to help with PYTHONPATH issues
import os
print('sys.path[0:5]=', sys.path[0:5])
print('PWD=', subprocess.run(['pwd'], capture_output=True, text=True).stdout.strip())


In [ ]:
# 3) If a GPU is present but torch.cuda is False, try installing a CUDA-compatible PyTorch wheel (cu121).
# This cell will attempt installation only when needed. It will ask you to restart the runtime if install runs.
import subprocess, json, sys
gpu_present = False
try:
    g = subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True)
    gpu_present = (g.returncode == 0 and bool(g.stdout.strip()))
except Exception:
    gpu_present = False
need_install = False
try:
    import torch
    if gpu_present and not torch.cuda.is_available():
        need_install = True
except Exception:
    # torch not installed -> attempt install if GPU present
    if gpu_present:
        need_install = True

print('GPU present:', gpu_present, 'PyTorch install/usable:', not need_install)
if need_install:
    print('Attempting to install CUDA-compatible PyTorch (cu121). This may take a few minutes.')
    cmds = [
        ['pip','uninstall','-y','torch','torchvision','torchaudio'],
        ['pip','cache','purge'],
        ['pip','install','--index-url','https://download.pytorch.org/whl/cu121','torch','torchvision','torchaudio','--upgrade']
    ]
    for c in cmds:
        print('Running:', ' '.join(c))
        r = subprocess.run(c)
        if r.returncode != 0:
            print('Command failed:', c, 'rc=', r.returncode)
    print('
Installation attempted. If pip installed new wheels, please Restart the runtime (Runtime -> Restart runtime) and re-run this notebook from the top.')


In [ ]:
# 4) Clone repo (shallow) or pull latest changes, and set PYTHONPATH for the notebook session
import os, subprocess, sys
repo_dir = '/content/openmontage-colab'
if not os.path.exists(repo_dir):
    print('Cloning repository into', repo_dir)
    subprocess.run(['git','clone','--depth','1','https://github.com/z3685507-tech/openmontage-colab.git',repo_dir])
else:
    print('Repository already exists; pulling latest changes')
    subprocess.run(['git','-C',repo_dir,'pull','--ff-only'], capture_output=True)
# Ensure PYTHONPATH points to the repo root for imports like `src.*`
os.environ['PYTHONPATH'] = repo_dir + ':' + os.environ.get('PYTHONPATH','')
print('PYTHONPATH set to', os.environ['PYTHONPATH'])
# change cwd to repo for running tools
os.chdir(repo_dir)
print('CWD now', os.getcwd())


In [ ]:
# 5) Run the strict Stage 1 smoke runner (may download models).
# OM_LOAD_ADAPTERS=1 allows adapters to import; OM_REAL_STRICT=1 enforces hard failure on missing adapters.
import os, subprocess
os.environ['OM_LOAD_ADAPTERS'] = '1'
os.environ['OM_REAL_STRICT'] = '1'
print('OM_LOAD_ADAPTERS=1 OM_REAL_STRICT=1')
# Run the smoke script and tee output to a file for later inspection.
ret = subprocess.run([sys.executable, 'tools/run_stage1_smoke.py'], capture_output=True, text=True)
print('Return code:', ret.returncode)
print('==== STDOUT ====')
print(ret.stdout)
print('==== STDERR ====')
print(ret.stderr)
# Show resulting report/log files (if any)
art = 'projects/smoke-report/artifacts'
for fname in ['smoke_report.json','real_run.log','mock_run.log']:
    p = os.path.join(art,fname)
    if os.path.exists(p):
        print('
---',fname,'---')
        print(open(p,'r',encoding='utf-8').read()[:20000])
    else:
        print('
(no file)',p)
